# NB3 -- Robustness Generator
## Computes robustness checks and saves to `Tables_datas/`

**Sierra-Porta & Varotsos (2025)** 

---
Run once. Produces:

| File | Contents |
|------|----------|
| `Tables_datas/robustness_interpolation.csv` | h(2), Delta-alpha under 0/5/10-day gap filling |
| `Tables_datas/robustness_detrending_ATHN.csv` | Rolling h(2), Delta-alpha with trend 180/365/730d (ATHN) |
| `Tables_datas/robustness_detrending_JUNG.csv` | Same for JUNG |

Effect size and bootstrap are computed in NB4 (no raw data needed).


## 0. Setup

In [1]:
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

import fathon
from fathon import fathonUtils as fu

os.makedirs('Tables_datas', exist_ok=True)
print(f'fathon {fathon.__version__} ready.')

# MF-DFA parameters (same as NB1)
WIN_SIZES_STATIC = fu.powRangeByCount(3, 10, 15, base=2)
WIN_SIZES_ROLL   = fu.powRangeByCount(3,  7, 10, base=2)
QS      = np.arange(-5, 5.1, 0.25)
REV_SEG = True
POL_ORD = 2
WINDOW_DAYS = 730
STEP_DAYS   = 1


fathon 1.3.3 ready.


## 1. Load raw data

In [2]:
df_raw = pd.read_csv(
    'daily_mean_NMDBallstations_missing_lessthan5_2010-2025.csv',
    parse_dates=True)
df_raw['start_date_time'] = pd.to_datetime(df_raw['start_date_time'])
df_raw.set_index('start_date_time', inplace=True)
if 'MOSC' in df_raw.columns:
    df_raw = df_raw.drop(columns=['MOSC'])

# Base clean: interpolate short gaps only (for subsequent tests)
df_base = df_raw.copy()
print(f'Raw data: {len(df_raw)} days, {len(df_raw.columns)} stations')
print(f'NaN count: {df_raw.isnull().sum().sum()} total')


Raw data: 5796 days, 16 stations
NaN count: 1797 total


## 2. MF-DFA helper functions

In [3]:
def run_mfdfa_static(series, win_sizes=WIN_SIZES_STATIC):
    x = np.array(series, dtype=float)
    x = fu.toAggregated(x)
    obj = fathon.MFDFA(x)
    n, F = obj.computeFlucVec(win_sizes, QS, revSeg=REV_SEG, polOrd=POL_ORD)
    list_H, _ = obj.fitFlucVec()
    alpha, f_alpha = obj.computeMultifractalSpectrum()
    h2 = list_H[np.argmin(np.abs(QS - 2))]
    valid = np.isfinite(alpha) & np.isfinite(f_alpha) & (f_alpha >= -0.5)
    da = alpha[valid].max() - alpha[valid].min() if valid.sum() >= 3 else np.nan
    return float(h2), float(da)


def rolling_mfdfa_custom(series, trend_window=365, window=WINDOW_DAYS,
                          step=STEP_DAYS, win_sizes=WIN_SIZES_ROLL):
    """
    Rolling MF-DFA with a configurable detrending window.
    Preprocessing: clip + detrend with trend_window + normalise.
    """
    # Step 0: clip outliers
    s = series.copy()
    roll_med = s.rolling(30, center=True, min_periods=10).median()
    roll_std = s.rolling(30, center=True, min_periods=10).std()
    mask = (s > roll_med + 3*roll_std) | (s < roll_med - 3*roll_std)
    s[mask] = np.nan
    s = s.interpolate(method='linear')
    # Step 1: detrend with specified window
    trend = s.rolling(trend_window, center=True, min_periods=int(trend_window*0.5))\
             .mean().ffill().bfill()
    detrended = s - trend
    # Step 2: normalise
    norm = detrended / s.mean() * 100

    x     = np.array(norm, dtype=float)
    dates = norm.index
    results = []
    starts = range(0, len(x) - window + 1, step)
    for i, start in enumerate(starts):
        seg = x[start:start + window]
        center_date = dates[start + window // 2]
        if np.isnan(seg).sum() > window * 0.05:
            results.append({'date': center_date, 'h2': np.nan, 'delta_alpha': np.nan})
            continue
        seg = pd.Series(seg).interpolate().values
        try:
            seg_agg = fu.toAggregated(seg)
            obj = fathon.MFDFA(seg_agg)
            n_, F_ = obj.computeFlucVec(win_sizes, QS, revSeg=REV_SEG, polOrd=POL_ORD)
            list_H_, _ = obj.fitFlucVec()
            alpha_, f_ = obj.computeMultifractalSpectrum()
            h2_ = list_H_[np.argmin(np.abs(QS - 2))]
            valid = np.isfinite(alpha_) & np.isfinite(f_) & (f_ >= -0.5)
            da_ = alpha_[valid].max() - alpha_[valid].min() \
                  if valid.sum() >= 3 else np.nan
            results.append({'date': center_date, 'h2': h2_, 'delta_alpha': da_})
        except Exception:
            results.append({'date': center_date, 'h2': np.nan, 'delta_alpha': np.nan})
        if (i+1) % 500 == 0:
            print(f'  {i+1}/{len(starts)} windows', end='\r')
    return pd.DataFrame(results).set_index('date')

print('Functions defined.')


Functions defined.


## 3. Robustness check 1: Interpolation sensitivity

We compare static MF-DFA results for ATHN and JUNG under three
gap-filling regimes:
- **Limit 0**: no interpolation (NaNs kept, passed as-is)
- **Limit 5**: linear interpolation up to 5 consecutive missing days
- **Limit 10**: linear interpolation up to 10 consecutive missing days (our choice)

If h(2) and Delta-alpha are stable across regimes, interpolation
does not drive the results.


In [4]:
out_file = 'Tables_datas/robustness_interpolation.csv'

if os.path.exists(out_file):
    print('Loaded from cache:', out_file)
else:

    def preprocess_with_limit(df_in, interp_limit, trend_window=365):
        """Preprocess CR series with variable gap-filling limit."""
        df_c = df_in.copy()
        # Step 1: gap filling (skip if limit=0)
        if interp_limit > 0:
            df_c = df_c.interpolate(method='linear', limit=interp_limit)
        # Step 2: outlier removal
        roll_med = df_c.rolling(30, center=True, min_periods=10).median()
        roll_std = df_c.rolling(30, center=True, min_periods=10).std()
        for col in df_c.columns:
            mask = (df_c[col] > roll_med[col] + 3*roll_std[col]) | (df_c[col] < roll_med[col] - 3*roll_std[col])
            df_c.loc[mask, col] = np.nan
            # Re-interpolate flagged points only if limit > 0
            if interp_limit > 0:
                df_c[col] = df_c[col].interpolate(method='linear', limit=interp_limit)
        # Step 3: detrend and normalise
        trend = df_c.rolling(trend_window, center=True, min_periods=180).mean().ffill().bfill()
        df_proc = (df_c - trend) / df_c.mean() * 100
        return df_proc

    rows = []
    for limit in [0, 5, 10]:
        print(f'Running limit={limit}...')
        df_proc = preprocess_with_limit(df_raw, limit)
        for st in ['ATHN', 'JUNG']:
            series = df_proc[st].dropna()
            try:
                h2, da = run_mfdfa_static(series)
            except Exception as e:
                h2, da = np.nan, np.nan
                print(f'  ERROR {st}: {e}')
            n_valid = int(df_proc[st].notna().sum())
            rows.append({
                'station': st,
                'interp_limit': limit,
                'h2': round(h2, 4),
                'delta_alpha': round(da, 4),
                'n_valid': n_valid
            })
            print(f'  {st}: h2={h2:.4f}  da={da:.4f}  N={n_valid}')

    df_interp = pd.DataFrame(rows)
    df_interp.to_csv(out_file, index=False)
    print(f'Saved: {out_file}')

df_interp = pd.read_csv(out_file)
print('=== Interpolation sensitivity ===')
print(df_interp.to_string(index=False))


Loaded from cache: Tables_datas/robustness_interpolation.csv
=== Interpolation sensitivity ===
station  interp_limit     h2  delta_alpha  n_valid
   ATHN             0 0.9858       0.4901     5532
   JUNG             0 0.9838       0.4275     5767
   ATHN             5 0.9877       0.9720     5715
   JUNG             5 0.9880       0.4516     5796
   ATHN            10 0.9964       1.2421     5744
   JUNG            10 0.9880       0.4516     5796


## 4. Robustness check 2: Detrending window sensitivity

We repeat the rolling MF-DFA on ATHN and JUNG using three
detrending windows:
- **180 days** (shorter: removes less solar-cycle structure)
- **365 days** (our choice -- reference)
- **730 days** (longer: matches the rolling MF-DFA window)

If the qualitative pattern (h(2) elevated at FDs, depressed
at solar minimum) is stable, the detrending choice is not critical.


In [5]:
FD1 = pd.Timestamp('2024-03-24')
FD2 = pd.Timestamp('2024-05-10')

for st in ['ATHN', 'JUNG']:
    out_file = f'Tables_datas/robustness_detrending_{st}.csv'
    if os.path.exists(out_file):
        print(f'{st}: loaded from cache')
        continue

    dfs = {}
    for tw in [180, 365, 730]:
        cache = f'Tables_datas/robustness_roll_{st}_trend{tw}.csv'
        if os.path.exists(cache):
            dfs[tw] = pd.read_csv(cache, index_col='date', parse_dates=True)
            print(f'  {st} trend={tw}d: loaded ({len(dfs[tw])} windows)')
        else:
            print(f'  Computing {st} trend={tw}d...')
            dfs[tw] = rolling_mfdfa_custom(df_raw[st], trend_window=tw)
            dfs[tw].to_csv(cache)
            print(f'  Done: {len(dfs[tw])} windows -> {cache}')

    # Merge into a single wide table
    combined = pd.DataFrame(index=dfs[365].index)
    for tw in [180, 365, 730]:
        combined[f'h2_trend{tw}']    = dfs[tw]['h2']
        combined[f'da_trend{tw}']    = dfs[tw]['delta_alpha']
    combined.to_csv(out_file)
    print(f'Saved: {out_file}')

print('\nDetrending robustness tables ready.')


  Computing ATHN trend=180d...
  Done: 5067 windows -> Tables_datas/robustness_roll_ATHN_trend180.csv
  Computing ATHN trend=365d...
  Done: 5067 windows -> Tables_datas/robustness_roll_ATHN_trend365.csv
  Computing ATHN trend=730d...
  Done: 5067 windows -> Tables_datas/robustness_roll_ATHN_trend730.csv
Saved: Tables_datas/robustness_detrending_ATHN.csv
  Computing JUNG trend=180d...
  Done: 5067 windows -> Tables_datas/robustness_roll_JUNG_trend180.csv
  Computing JUNG trend=365d...
  Done: 5067 windows -> Tables_datas/robustness_roll_JUNG_trend365.csv
  Computing JUNG trend=730d...
  Done: 5067 windows -> Tables_datas/robustness_roll_JUNG_trend730.csv
Saved: Tables_datas/robustness_detrending_JUNG.csv

Detrending robustness tables ready.


## 5. Summary

In [6]:
print('=== Tables_datas/ robustness files ===')
for f in sorted(os.listdir('Tables_datas')):
    if 'robust' in f:
        kb = os.path.getsize(f'Tables_datas/{f}') / 1024
        print(f'  {f:<55s} {kb:6.1f} KB')
print('\nRun NB4_RobustnessAnalysis to produce tables and figures.')


=== Tables_datas/ robustness files ===
  robustness_detrending_ATHN.csv                           612.1 KB
  robustness_detrending_JUNG.csv                           616.1 KB
  robustness_interpolation.csv                               0.2 KB
  robustness_roll_ATHN_trend180.csv                        240.4 KB
  robustness_roll_ATHN_trend365.csv                        240.3 KB
  robustness_roll_ATHN_trend730.csv                        240.2 KB
  robustness_roll_JUNG_trend180.csv                        241.7 KB
  robustness_roll_JUNG_trend365.csv                        241.7 KB
  robustness_roll_JUNG_trend730.csv                        241.6 KB

Run NB4_RobustnessAnalysis to produce tables and figures.
